# The Golden Spiral and Dürer's Eve
## Pedagogical notebook for instructors — Advanced level

This notebook accompanies the article:
> Galeano J R (2026) *Art and Physics at the Museo del Prado: A Walk Through Science Hidden in Masterpieces*. European Journal of Physics.

It supports the **advanced level** of Block 3 in Section 5 (Dürer), providing a quantitative test of whether the golden spiral fits the composition of Dürer's *Eve* better than a non-golden logarithmic spiral.

### What this notebook does
1. Loads the official high-resolution image of *Eve* from the Museo del Prado.
2. Superimposes a golden spiral on the image using the parametric equations from the article.
3. Asks the student to identify anatomical landmark coordinates.
4. Calculates the root-mean-square deviation (RMSD) between the spiral and the landmarks.
5. Compares the golden spiral ($b = 2\ln\Phi/\pi$) with non-golden logarithmic spirals.

**Image source:** Download the official high-resolution image of *Eve* from:
https://www.museodelprado.es/coleccion/obra-de-arte/eva/930c0fdf-fcfc-47df-b216-e375f5719084
and save it as `Eva.jpg` in the same folder as this notebook.

## 0. Imports and constants

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from scipy.optimize import minimize
import scipy.constants as const

# Golden ratio (exact)
Phi = (1 + np.sqrt(5)) / 2

# Golden spiral growth rate (b such that r grows by Phi every quarter-turn)
b_golden = 2 * np.log(Phi) / np.pi

print(f"Golden ratio:          Φ = {Phi:.6f}")
print(f"Golden spiral growth:  b = 2·ln(Φ)/π = {b_golden:.6f}")
print(f"Verification: e^(b·π/2) = {np.exp(b_golden * np.pi / 2):.6f} (should equal Φ)")

## 1. Load the image of Eve

The image coordinate system in matplotlib has the origin at the **top-left** corner, with x increasing rightward and y increasing downward. We keep this convention throughout the notebook.

In [ ]:
# Load the image
try:
    img = mpimg.imread('Eva.jpg')
    height, width = img.shape[:2]
    print(f"Image loaded: {width} × {height} pixels")
except FileNotFoundError:
    print("ERROR: Eva.jpg not found.")
    print("Please download the image from the Museo del Prado website and")
    print("save it as 'Eva.jpg' in the same folder as this notebook.")
    raise

# Display the image
fig, ax = plt.subplots(figsize=(5, 12))
ax.imshow(img)
ax.set_title("Albrecht Dürer, Eve (1507)\nMuseo del Prado, Madrid", fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.savefig('eve_original.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Define the logarithmic spiral

The logarithmic spiral in Cartesian coordinates is:

$$x(\theta) = x_0 + a\,e^{b\theta}\cos\theta$$
$$y(\theta) = y_0 + a\,e^{b\theta}\sin\theta$$

where $(x_0, y_0)$ is the centre of the spiral, $a$ is a scaling factor, $b$ is the growth rate, and $\theta$ is the polar angle.

For the **golden spiral**, $b = 2\ln\Phi/\pi \approx 0.306$.

In [ ]:
def spiral_xy(theta, x0, y0, a, b):
    """
    Logarithmic spiral in Cartesian coordinates.

    Parameters
    ----------
    theta : array
        Polar angle in radians.
    x0, y0 : float
        Centre of the spiral in pixel coordinates.
    a : float
        Scaling factor (initial radius at theta=0).
    b : float
        Growth rate. For the golden spiral: b = 2*ln(Phi)/pi.

    Returns
    -------
    x, y : arrays
        Cartesian coordinates of the spiral.
    """
    r = a * np.exp(b * theta)
    x = x0 + r * np.cos(theta)
    y = y0 + r * np.sin(theta)
    return x, y


def rmsd(landmarks, x0, y0, a, b, theta_range):
    """
    Root-mean-square deviation between the spiral and a set of landmark points.

    For each landmark, we find the point on the spiral closest to it
    and compute the Euclidean distance. The RMSD is the square root
    of the mean squared distance over all landmarks.

    Parameters
    ----------
    landmarks : list of (x, y) tuples
        Pixel coordinates of the anatomical landmarks.
    x0, y0, a, b : float
        Spiral parameters.
    theta_range : tuple (theta_min, theta_max)
        Angular range of the spiral.

    Returns
    -------
    float
        RMSD in pixels.
    """
    theta = np.linspace(theta_range[0], theta_range[1], 10000)
    sx, sy = spiral_xy(theta, x0, y0, a, b)

    distances = []
    for lx, ly in landmarks:
        d = np.sqrt((sx - lx)**2 + (sy - ly)**2)
        distances.append(np.min(d))

    return np.sqrt(np.mean(np.array(distances)**2))


print("Functions defined: spiral_xy() and rmsd()")

## 3. Define the anatomical landmarks

Students identify key anatomical points on the image and record their pixel coordinates.
The easiest way is to use the interactive plot below: click on the image to record coordinates.

We provide approximate coordinates measured from the Prado high-resolution image as a reference.
Students should adjust these to match their own downloaded image.

**Landmarks (approximate pixel coordinates for a ~1500×3500 px image):**
- Apple (centre)
- Right wrist
- Right shoulder
- Left breast
- Left side of chest
- Navel

In [ ]:
# ----------------------------------------------------------------
# STUDENT TASK: adjust these coordinates to match your image.
# Use the interactive plot below to identify the pixel coordinates
# of each anatomical landmark.
# Format: (x_pixel, y_pixel) with origin at top-left corner.
# ----------------------------------------------------------------

landmarks = {
    'Apple':           (1100, 820),   # centre of the apple
    'Right wrist':     (1050, 950),   # right wrist holding apple
    'Right shoulder':  ( 950, 600),   # right shoulder
    'Left breast':     ( 680, 900),   # left breast
    'Left chest':      ( 550, 980),   # left side of chest
    'Navel':           ( 730, 1350),  # navel
}

# Display image with landmarks
fig, ax = plt.subplots(figsize=(5, 12))
ax.imshow(img)
for name, (lx, ly) in landmarks.items():
    ax.plot(lx, ly, 'ro', markersize=8)
    ax.annotate(name, (lx, ly), textcoords='offset points',
                xytext=(8, 0), fontsize=8, color='red')
ax.set_title("Anatomical landmarks", fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.savefig('eve_landmarks.png', dpi=150, bbox_inches='tight')
plt.show()

landmark_list = list(landmarks.values())
print(f"Number of landmarks: {len(landmark_list)}")
for name, coords in landmarks.items():
    print(f"  {name}: {coords}")

### 3.1 Interactive landmark selection (optional)

Run this cell to click on the image and record pixel coordinates interactively.
Press **Enter** when done.

In [ ]:
# Uncomment and run this cell to select landmarks interactively
# %matplotlib notebook
#
# fig, ax = plt.subplots(figsize=(5, 12))
# ax.imshow(img)
# ax.set_title("Click on anatomical landmarks. Press Enter when done.", fontsize=10)
# ax.axis('off')
# clicked_points = plt.ginput(n=-1, timeout=0)
# plt.show()
# print("Selected points:", clicked_points)

## 4. Fit the golden spiral to the image

We set the spiral parameters manually to match the GeoGebra construction:
- Centre $(x_0, y_0)$: near the remaining small rectangle after the three subdivisions.
- Scaling factor $a$: adjusted so the spiral passes through the apple.
- Angular range: covering approximately $3\pi/2$ radians (three quarter-turns).

Students should adjust these parameters to best match their image.

In [ ]:
# ----------------------------------------------------------------
# STUDENT TASK: adjust these parameters to match your image.
# ----------------------------------------------------------------

# Spiral parameters (approximate, for a ~1500×3500 px image)
x0    =  820    # centre x (pixel)
y0    =  640    # centre y (pixel)
a     =  280    # scaling factor
b     = b_golden  # golden spiral
theta_min = np.pi        # start angle
theta_max = 3 * np.pi    # end angle (two full turns from theta_min)

# Generate spiral
theta = np.linspace(theta_min, theta_max, 2000)
sx, sy = spiral_xy(theta, x0, y0, a, b)

# Plot
fig, ax = plt.subplots(figsize=(5, 12))
ax.imshow(img)
ax.plot(sx, sy, color='teal', linewidth=2.5, label=f'Golden spiral (b={b:.3f})')
for name, (lx, ly) in landmarks.items():
    ax.plot(lx, ly, 'ro', markersize=8)
    ax.annotate(name, (lx, ly), textcoords='offset points',
                xytext=(8, 0), fontsize=8, color='red')
ax.plot(x0, y0, 'w+', markersize=12, markeredgewidth=2, label='Spiral centre')
ax.legend(fontsize=9, loc='lower left')
ax.set_title(f"Golden spiral superimposed on Dürer's Eve\n"
             f"b = 2·ln(Φ)/π = {b:.4f}", fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.savefig('eve_golden_spiral.png', dpi=150, bbox_inches='tight')
plt.show()

# Compute RMSD
rms = rmsd(landmark_list, x0, y0, a, b, (theta_min, theta_max))
print(f"RMSD (golden spiral, b={b:.4f}): {rms:.1f} pixels")

## 5. Compare with non-golden logarithmic spirals

**Question 3 from the article:** Could a logarithmic spiral with a different growth rate $b$ fit the composition equally well?

We compare the RMSD of the golden spiral with spirals having different values of $b$.

In [ ]:
# Range of b values to test
b_values = np.linspace(0.1, 0.6, 50)
rmsd_values = [rmsd(landmark_list, x0, y0, a, bv, 
                    (theta_min, theta_max)) for bv in b_values]

# Find the b that minimises RMSD
idx_min = np.argmin(rmsd_values)
b_best  = b_values[idx_min]
rmsd_best = rmsd_values[idx_min]
rmsd_golden = rmsd(landmark_list, x0, y0, a, b_golden,
                   (theta_min, theta_max))

# Plot RMSD vs b
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(b_values, rmsd_values, 'k-', linewidth=2)
ax.axvline(b_golden, color='teal', linestyle='--', linewidth=2,
           label=f'Golden spiral: b = {b_golden:.3f}')
ax.axvline(b_best, color='red', linestyle=':', linewidth=2,
           label=f'Best fit: b = {b_best:.3f} (RMSD = {rmsd_best:.1f} px)')
ax.set_xlabel('Growth rate b', fontsize=12)
ax.set_ylabel('RMSD (pixels)', fontsize=12)
ax.set_title('RMSD between logarithmic spiral and anatomical landmarks\n'
             'as a function of growth rate b', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('eve_rmsd_vs_b.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Golden spiral:  b = {b_golden:.4f},  RMSD = {rmsd_golden:.1f} px")
print(f"Best-fit spiral: b = {b_best:.4f},  RMSD = {rmsd_best:.1f} px")
print()
if abs(b_best - b_golden) < 0.05:
    print("The best-fit b is close to the golden spiral value.")
    print("This supports (but does not prove) the golden spiral hypothesis.")
else:
    print(f"The best-fit b ({b_best:.3f}) differs from the golden value ({b_golden:.3f}).")
    print("A non-golden spiral fits the landmarks better with these anchor points.")
    print("This illustrates the sensitivity of the result to the choice of landmarks.")

## 6. Visualise the best-fit spiral alongside the golden spiral

In [ ]:
# Generate best-fit spiral
sx_best, sy_best = spiral_xy(theta, x0, y0, a, b_best)

fig, axes = plt.subplots(1, 2, figsize=(10, 12))

for ax, (bv, sx_plot, sy_plot, label, col) in zip(
    axes,
    [
        (b_golden, sx, sy,
         f'Golden spiral\nb = {b_golden:.3f}', 'teal'),
        (b_best, sx_best, sy_best,
         f'Best-fit spiral\nb = {b_best:.3f}', 'red'),
    ]
):
    ax.imshow(img)
    ax.plot(sx_plot, sy_plot, color=col, linewidth=2.5, label=label)
    for name, (lx, ly) in landmarks.items():
        ax.plot(lx, ly, 'wo', markersize=7,
                markeredgecolor='black', markeredgewidth=1)
    rms_val = rmsd(landmark_list, x0, y0, a, bv, (theta_min, theta_max))
    ax.set_title(f"{label}\nRMSD = {rms_val:.1f} px", fontsize=10)
    ax.legend(fontsize=9, loc='lower left')
    ax.axis('off')

plt.suptitle("Comparison: golden spiral vs best-fit spiral\n"
             "White dots: anatomical landmarks", fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('eve_spiral_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Discussion questions

After running the notebook, students answer the following questions:

1. **Does the golden spiral pass convincingly through the anatomical landmarks?**  
   Look at the RMSD value. Is it small relative to the size of the figure in the image?

2. **Is the golden spiral the best-fitting logarithmic spiral?**  
   Compare $b_{\rm golden}$ with $b_{\rm best}$. If they are similar, the data supports (but does not prove) the golden spiral hypothesis. If they differ significantly, a non-golden spiral fits better.

3. **How sensitive is the result to the choice of landmarks?**  
   Re-run the notebook with a different set of anatomical points. Does $b_{\rm best}$ change? What does this tell you about the robustness of the claim?

4. **What can you conclude about Dürer's intentional use of the golden spiral?**  
   Remember that a good fit is necessary but not sufficient evidence for intentional use. What additional evidence would you need?

---
## References

- Galeano J R (2026) *Art and Physics at the Museo del Prado*. European Journal of Physics.
- Markowsky G (1992) Misconceptions about the golden ratio. *The College Mathematics Journal* **23** 2–19.
- Dürer A (1528) *Vier Bücher von menschlicher Proportion*. Nuremberg.
- Museo Nacional del Prado (2024) *Eve* [Digital image]. https://www.museodelprado.es

---
*Notebook by Javier R. Galeano, Universidad Politécnica de Madrid, 2026.*  
*Available at: https://github.com/jgaleano/prado-physics*  
*Licence: CC BY 4.0*